# 11 — DuckDB : SQL analytique sur fichiers

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre pourquoi DuckDB existe (OLAP embarqué)
- requêter directement des fichiers CSV et Parquet
- intégrer avec Pandas et Polars
- utiliser les fonctions d'agrégation avancées

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- toute la formation Intermédiaire
- `Decimal`, `Fraction`, `statistics`, `random`
- NumPy, Pandas, Polars, matplotlib
- SQL avec sqlite3 (jour 4)

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- rien de spécifique

## Plan

1. Pourquoi DuckDB
2. Installation et premier contact
3. Requêter un CSV/Parquet
4. Intégration Pandas et Polars
5. Fonctions d'agrégation avancées
6. DuckDB vs SQLite
7. Synthèse
8. Exercices

---

## 1. Pourquoi DuckDB

DuckDB est un moteur SQL **analytique** (OLAP), pensé pour les **requêtes sur colonnes** à grande échelle, directement en mémoire ou sur des fichiers. C'est le « SQLite de l'analytique ».

---

## 2. Installation et premier contact

In [ ]:
import duckdb

duckdb.sql('SELECT 42 AS answer').show()


---

## 3. Requêter un CSV/Parquet

DuckDB lit les fichiers directement, sans chargement explicite.

In [ ]:
# Créons un CSV pour la démo
import csv
with open('/tmp/salles.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['nom', 'capacite', 'prix_h'])
    w.writerows([['Mars', 12, 50], ['Venus', 6, 30], ['Io', 15, 80]])


In [ ]:
duckdb.sql("SELECT * FROM '/tmp/salles.csv'").show()


In [ ]:
duckdb.sql("SELECT nom, prix_h * 1.2 AS prix_ttc FROM '/tmp/salles.csv' WHERE capacite > 8").show()


---

## 4. Intégration Pandas et Polars

In [ ]:
import pandas as pd

df = pd.DataFrame({'salle': ['Mars', 'Venus', 'Mars'], 'heures': [2, 1, 3]})
duckdb.sql('SELECT salle, SUM(heures) as total FROM df GROUP BY salle').show()


In [ ]:
import polars as pl

pldf = pl.DataFrame({'x': [1, 2, 3], 'y': [10, 20, 30]})
duckdb.sql('SELECT x, y, x * y as produit FROM pldf').show()


---

## 5. Fonctions d'agrégation avancées

In [ ]:
duckdb.sql('''
    SELECT nom, capacite,
           RANK() OVER (ORDER BY capacite DESC) AS rang
    FROM '/tmp/salles.csv'
''').show()


---

## 6. DuckDB vs SQLite

| Critère | SQLite | DuckDB |
|---|---|---|
| Optimisé pour | OLTP (transactions) | OLAP (analytique) |
| Stockage | Ligne par ligne | Colonne par colonne |
| Lit des fichiers directement | Non | CSV, Parquet, JSON |
| Intégration Pandas/Polars | Manuelle | Native |
| Usage typique | App mobile, serveur web | Data science, ETL |


---

## Synthèse

| API | Rôle |
|---|---|
| `duckdb.sql('...')` | Requête SQL |
| `FROM 'fichier.csv'` | Lecture directe de fichiers |
| `FROM df` | Requête sur DataFrame Pandas/Polars |
| Window functions | `RANK()`, `LAG()`, `LEAD()` |


### Règles à retenir

1. **DuckDB pour l'analytique**, SQLite pour le transactionnel.
2. **Lecture directe de fichiers** : pas besoin de charger en mémoire d'abord.
3. **Intégration native** avec les DataFrames : SQL sur Pandas/Polars.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Requête sur CSV *(facile)*

Écrire un CSV de 5 produits, puis une requête DuckDB pour les trier par prix décroissant.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la correction</summary>

```python
import duckdb, csv
with open('/tmp/produits.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['nom', 'prix'])
    w.writerows([['A', 10], ['B', 25], ['C', 5], ['D', 40], ['E', 15]])

duckdb.sql("SELECT * FROM '/tmp/produits.csv' ORDER BY prix DESC").show()
```

</details>

### Exercice 2 — SQL sur DataFrame *(moyen)*

Créer un DataFrame Pandas de ventes (produit, region, montant). Utiliser DuckDB pour calculer le total par région.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la correction</summary>

```python
import pandas as pd, duckdb
df = pd.DataFrame({'produit': ['A','A','B'], 'region': ['N','S','N'], 'montant': [100,200,150]})
duckdb.sql('SELECT region, SUM(montant) FROM df GROUP BY region').show()
```

</details>

### Exercice 3 — Window function *(difficile)*

À partir d'un CSV de ventes mensuelles, calculer le total cumulé par mois avec une window function.

In [ ]:
# Votre code ici


<details>
<summary>📖 Voir la correction</summary>

```python
import duckdb, csv
with open('/tmp/ventes.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['mois', 'montant'])
    for i, m in enumerate([100, 150, 200, 120, 180], 1):
        w.writerow([i, m])

duckdb.sql('''
    SELECT mois, montant,
           SUM(montant) OVER (ORDER BY mois) AS cumul
    FROM '/tmp/ventes.csv'
''').show()
```

</details>

---

## Ressources externes

### Documentation officielle
- [DuckDB docs](https://duckdb.org/docs/)